# M6 â€” Model Evaluation & Hyperparameter Tuning (GridSearchCV)
**Dataset: Breast Cancer Wisconsin (sklearn) â€” 4 Model: Decision Tree, KNN, Logistic Regression, XGBoost**

Tugas praktikum: evaluasi model dengan 5-Fold Stratified CV (Accuracy, Precision, Recall, F1),
tuning dengan GridSearchCV (scoring=F1), perbandingan Default vs Tuned, dan evaluasi final pada test set + Confusion Matrix.

Ketentuan yang dipenuhi notebook ini:
- Split 80/20, `stratify=y`, `random_state=2020`
- 5-Fold Stratified Cross-Validation untuk evaluasi & tuning
- Minimal 1 hyperparameter & minimal 5 nilai per model, scoring utama F1
- Test set TIDAK dipakai selama training/tuning, hanya untuk evaluasi final
- KNN & Logistic Regression memakai `Pipeline(StandardScaler)` agar adil (fitur kanker beda skala)
- Satu file ini berisi kode + tabel/grafik + interpretasi (Aâ€“H + Kesimpulan)

In [ ]:
# --- Setup ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost belum terinstal. Di Kaggle umumnya sudah ada. Jika perlu, jalankan: !pip install -q xgboost")

print("Setup OK. HAS_XGB =", HAS_XGB)

## A. Data Preparation (poin 1â€“4)
- Bagi 80% train / 20% test, `stratify=y`, `random_state=2020`
- Tampilkan ukuran train/test dan jumlah tiap kelas
- Target: 0 = malignant (ganas), 1 = benign (jinak)

In [ ]:
# --- A. Data Preparation ---
data = load_breast_cancer()
X, y = data.data, data.target
print("Fitur:", list(data.feature_names[:5]), "... total", X.shape[1])
print("Target:", list(data.target_names), "| Total observasi:", X.shape[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=2020)

print("\nUkuran train:", X_train.shape, "| Ukuran test:", X_test.shape)
train_counts = np.bincount(y_train); test_counts = np.bincount(y_test)
print(f"Train -> malignant(0): {train_counts[0]}, benign(1): {train_counts[1]}")
print(f"Test  -> malignant(0): {test_counts[0]}, benign(1): {test_counts[1]}")
print(f"Proporsi train benign: {train_counts[1]/len(y_train):.3f} | test benign: {test_counts[1]/len(y_test):.3f} (stratify menjaga proporsi)")

## B. Pemilihan Model (poin 5â€“7)
**Model yang dipilih (4 model, masing-masing dikerjakan full Aâ€“H):**
1. **Decision Tree** â€” baseline non-linear yang interpretable (aturan if-then mirip alur diagnosis).
2. **KNN** â€” instance-based; cocok sebagai pembanding karena sensitif terhadap skala fitur â†’ wajib scaler.
3. **Logistic Regression** â€” baseline linear probabilistik yang kuat untuk klasifikasi biner.
4. **XGBoost** â€” gradient boosting; pembanding modern yang biasanya kuat di data tabular.

Semua model pertama dipakai dengan **default hyperparameter** (kecuali `random_state=2020` agar reproducible,
dan Pipeline scaler untuk KNN/LogReg).

In [ ]:
# --- B. Definisi 4 model default ---
models_default = {
    "DecisionTree": DecisionTreeClassifier(random_state=2020),
    "KNN": Pipeline([("scaler", StandardScaler()),
                     ("clf", KNeighborsClassifier())]),
    "LogReg": Pipeline([("scaler", StandardScaler()),
                        ("clf", LogisticRegression(max_iter=5000))]),
}
if HAS_XGB:
    models_default["XGBoost"] = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=2020, n_jobs=-1)

for name, m in models_default.items():
    print(f"{name}: {m}")

## C. Model Evaluation dengan Cross-Validation (poin 8â€“9)
Evaluasi model **default** memakai **5-Fold Stratified CV pada data training saja**.
Metrik: Accuracy, Precision, Recall, F1-score.

In [ ]:
# --- C. Evaluasi default: 5-Fold Stratified CV ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2020)
scoring = ["accuracy", "precision", "recall", "f1"]

def cv_report(model, X_tr, y_tr):
    r = cross_validate(model, X_tr, y_tr, cv=cv, scoring=scoring)
    return {m: float(r[f"test_{m}"].mean()) for m in scoring}

rows = []
for name, m in models_default.items():
    s = cv_report(m, X_train, y_train)
    rows.append({"Model": name, **{k: round(v, 4) for k, v in s.items()}})

df_default_cv = pd.DataFrame(rows).set_index("Model")
print(df_default_cv.to_string())
df_default_cv

### Jawaban 8 â€” Bagaimana performa model default (CV)?
Hasil referensi (local run, sklearn 1.7.2 â€” di Kaggle bisa beda tipis di desimal akhir):
- **LogReg**: Acc 0.9802, Prec 0.9826, Rec 0.9860, F1 **0.9842** (terbaik di semua metrik)
- **KNN**: Acc 0.9648, Prec 0.9536, Rec **0.9930**, F1 0.9727 (recall tertinggi)
- **XGBoost**: Acc 0.9560, Prec 0.9624, Rec 0.9684, F1 0.9650
- **DecisionTree**: Acc 0.9275, Prec 0.9432, Rec 0.9439, F1 0.9426 (terendah, single tree variance tinggi)

### Jawaban 9 â€” Metrik mana paling relevan?
**Recall dan F1.** Pada kanker, False Negative (pasien ganas diprediksi jinak) berakibat fatal
(terlambat ditangani), jadi Recall (menangkap sebanyak mungkin kasus benign... tepatnya kelas positif=jinak;
dalam praktik klinis label dibalik â€” yang kritis adalah jangan sampai kasus ganas lolos) harus tinggi.
Namun Precision juga penting agar tidak banyak false alarm. **F1** (harmonic mean Precisionâ€“Recall)
dipakai sebagai scoring utama GridSearchCV karena menyeimbangkan keduanya di data yang sedikit imbalance
(357 jinak vs 212 ganas). Accuracy saja tidak cukup karena bisa menutupi kegagalan di kelas minoritas.

## D. Hyperparameter Tuning dengan GridSearchCV
- Scoring utama: **F1**, CV: **5-Fold Stratified**
- Minimal 1 hyperparameter & â‰¥5 nilai per model (lihat grid di bawah)
- Tampilkan `best_params_`, `best_score_`, dan hasil tiap kombinasi

In [ ]:
# --- D. GridSearchCV (scoring='f1', cv=5) ---
param_grids = {
    "DecisionTree": {"max_depth": [3, 5, 7, 10, 15, None]},          # 6 nilai
    "KNN":          {"clf__n_neighbors": [3, 5, 7, 9, 11, 15]},       # 6 nilai
    "LogReg":       {"clf__C": [0.01, 0.1, 1, 10, 100]},              # 5 nilai (log-scale)
}
if HAS_XGB:
    param_grids["XGBoost"] = {"max_depth": [3, 4, 5, 6, 7]}           # 5 nilai (n_estimators=200 fixed)

grid_results, best_estimators = {}, {}
for name in models_default:
    gs = GridSearchCV(estimator=models_default[name], param_grid=param_grids[name],
                      cv=cv, scoring="f1", n_jobs=-1)
    gs.fit(X_train, y_train)
    grid_results[name] = gs
    best_estimators[name] = gs.best_estimator_
    print(f"\n{name} | best_params_={gs.best_params_} | best_score_(F1 CV)={gs.best_score_:.4f}")
    cvres = pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score"]].sort_values("mean_test_score", ascending=False)
    print(cvres.to_string(index=False))

In [ ]:
# --- D. Grafik: F1 CV vs nilai hyperparameter ---
fig, axes = plt.subplots(1, len(grid_results), figsize=(5*len(grid_results), 4), squeeze=False)
axes = axes.ravel()
for ax, name in zip(axes, grid_results):
    gs = grid_results[name]
    means = gs.cv_results_["mean_test_score"]; stds = gs.cv_results_["std_test_score"]
    # label sumbu-x = nilai hyperparameter yang dituning
    key = list(param_grids[name].keys())[0]
    labels = [str(p[key]) for p in gs.cv_results_["params"]]
    x = np.arange(len(labels))
    ax.errorbar(x, means, yerr=stds, marker="o", capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30)
    ax.set_title(f"{name}: F1 vs {key}")
    ax.set_ylabel("mean F1 (5-fold)"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## E. Analisis Hasil GridSearchCV (poin 10â€“13)
### Jawaban 10 â€” Hyperparameter & nilai dengan F1 CV tertinggi
- **DecisionTree** `max_depth=7` â†’ F1 CV **0.9478**
- **KNN** `clf__n_neighbors=3` â†’ F1 CV **0.9761**
- **LogReg** `clf__C=1` â†’ F1 CV **0.9842** (sama dengan default karena default C=1.0)
- **XGBoost** `max_depth=5` â†’ F1 CV **0.9651** (nilai 5/6/7 plateau sama)

### Jawaban 11 â€” Bagaimana F1 berubah saat nilai berubah?
- **DecisionTree**: naik 3â†’7 (0.9328â†’0.9458â†’0.9478), lalu turun/plateau di 10/15/None (0.9426). Pola underfit â†’ optimal â†’ overfit.
- **KNN**: tertinggi di k=3 (0.9761), cenderung turun saat k membesar (k=15: 0.9642). k kecil = fleksibel, k besar = oversmoothing.
- **LogReg**: parabola â€” C=0.01 underfit (0.9597, regularisasi terlalu kuat), puncak di C=1 (0.9842), turun di C=10/100 (0.9754/0.9628, terlalu bebas/noise diikuti).
- **XGBoost**: naik 3â†’5 lalu datar (0.9633â†’0.9632â†’0.9651â†’0.9651â†’0.9651). Depth 5 cukup; depth lebih dalam tidak menambah sinyal.

### Jawaban 12 â€” Kesimpulan CV
Grid menunjukkan tiap model punya titik sweet-spot; tuning memberi gain kecil tapi konsisten
(kecuali LogReg yang memang sudah optimal). Std antar-fold kecil (Â±0.005â€“0.017) â†’ hasil stabil.

### Jawaban 13 â€” Mengapa CV dipakai dalam tuning?
Agar pemilihan hyperparameter tidak overfit ke satu split train/validasi. CV menguji tiap kandidat
di 5 lipatan berbeda lalu merata-ratakan â†’ estimasi generalisasi lebih jujur & stabil,
dan seluruh data training tetap terpakai (efisien). Test set tetap steril untuk evaluasi final.

## F. Perbandingan Default vs Tuned (poin 14â€“16)
Tuned model = estimator dengan hyperparameter terbaik dari GridSearchCV, dievaluasi ulang dengan CV yang sama.

In [ ]:
# --- F. Perbandingan Default vs Tuned (CV yang sama) ---
comp = []
for name in models_default:
    d = cv_report(models_default[name], X_train, y_train)
    t = cv_report(best_estimators[name], X_train, y_train)
    comp.append({"Model": name+" (Default)", **{k: round(v,4) for k,v in d.items()}})
    comp.append({"Model": name+" (Tuned)", **{k: round(v,4) for k,v in t.items()}})
df_comp = pd.DataFrame(comp).set_index("Model")
print(df_comp.to_string())
df_comp

### Jawaban 14â€“16
- **14. Apakah tuned berbeda dari default?** Ya untuk 3 model, tidak untuk LogReg.
  DT: F1 0.9426â†’**0.9478** (+0.0052); KNN: 0.9727â†’**0.9761** (+0.0034);
  XGB: 0.9650â†’**0.9651** (+0.0001, praktis sama); LogReg: **tetap 0.9842** (best C=1 = default).
- **15. Metrik apa yang berubah?** Semua metrik ikut bergerak; gain terbesar di F1 & Recall
  (DT: Recall 0.9439â†’0.9509, Acc 0.9275â†’0.9341).
- **16. Apakah tuning meningkatkan F1?** Ya, kecuali LogReg yang sudah optimal.
  Kenaikan kecil karena default-nya memang sudah kuat di dataset ini â€” tuning berperan sebagai
  fine-tuning + bukti bahwa default bukan kebetulan terbaik (khususnya DT & KNN).

## G. Final Evaluation pada Test Set
Test set (114 data: 42 malignant, 72 benign) **baru dipakai di sini** â€” tidak pernah dipakai saat training/tuning.

In [ ]:
# --- G. Evaluasi final tuned model pada TEST SET + Confusion Matrix ---
test_rows = []
fig, axes = plt.subplots(1, len(best_estimators), figsize=(5*len(best_estimators), 4), squeeze=False)
axes = axes.ravel()
for ax, name in zip(axes, best_estimators):
    est = best_estimators[name]
    pred = est.predict(X_test)
    acc = accuracy_score(y_test, pred); prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred); f1 = f1_score(y_test, pred)
    cm = confusion_matrix(y_test, pred)
    test_rows.append({"Model": name, "Accuracy": round(acc,4), "Precision": round(prec,4),
                      "Recall": round(rec,4), "F1": round(f1,4)})
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"{name} (test F1={f1:.4f})")
df_test = pd.DataFrame(test_rows).set_index("Model")
print(df_test.to_string())
plt.tight_layout(); plt.show()
df_test

### Interpretasi test set (hasil referensi)
| Model | Acc | Prec | Rec | F1 | Confusion Matrix [[TN,FP],[FN,TP]] |
|---|---|---|---|---|---|
| DecisionTree | 0.9298 | 0.9706 | 0.9167 | 0.9429 | [[40,2],[6,66]] |
| KNN | 0.9649 | 0.9595 | 0.9861 | 0.9726 | [[39,3],[1,71]] |
| LogReg | **0.9737** | 0.9726 | **0.9861** | **0.9793** | [[40,2],[1,71]] |
| XGBoost | 0.9561 | **0.9718** | 0.9583 | 0.9650 | [[40,2],[3,69]] |

- **Generalisasi baik**: F1 test â‰ˆ F1 CV (selisih â‰¤0.005) â†’ tidak overfit ke CV.
- **LogReg terbaik** di test (F1 0.9793, hanya 1 FN + 2 FP).
- **KNN** recall 0.9861 (hanya 1 FN) â€” bagus untuk skrining, tapi FP=3.
- **DecisionTree** recall terendah (6 FN) â€” risiko klinis paling tinggi meski precision-nya tinggi.
- **XGBoost** seimbang (3 FN, 2 FP, F1 0.9650). Secara keseluruhan urutan test: LogReg > KNN > XGBoost > DecisionTree.

## H. Kesimpulan
1. Dipakai 4 model pada Breast Cancer Wisconsin (455 train / 114 test, stratify, random_state=2020):
   Decision Tree, KNN (Pipeline+Scaler), Logistic Regression (Pipeline+Scaler), dan XGBoost.
2. Performa default (5-fold Stratified CV, F1): LogReg 0.9842 > KNN 0.9727 > XGBoost 0.9650 > DecisionTree 0.9426;
   KNN punya recall tertinggi (0.9930) sedangkan DecisionTree terendah di semua metrik.
3. Hyperparameter yang dituning (GridSearchCV, scoring F1): `max_depth` (DT), `n_neighbors` (KNN),
   `C` (LogReg), dan `max_depth` (XGBoost) â€” masing-masing â‰¥5 nilai.
4. Nilai terbaik: DT `max_depth=7` (F1 0.9478), KNN `n_neighbors=3` (0.9761),
   LogReg `C=1` (0.9842, sama dengan default), XGBoost `max_depth=5` (0.9651).
5. Tuned vs default (CV): DT +0.0052 dan KNN +0.0034 membaik, XGBoost praktis sama (+0.0001),
   LogReg tetap â€” tuning terbukti berguna sebagai fine-tuning.
6. Evaluasi final test set konsisten dengan CV (LogReg F1 0.9793, KNN 0.9726, XGBoost 0.9650, DT 0.9429),
   sehingga model â€” terutama Logistic Regression â€” layak dipilih untuk kasus ini.